# 02 — Khí hậu ERA5-Land (Copernicus CDS)

Luồng A3+A4 — [PHASE-1-CHECKLIST.md §A3-A4](../../PHASE-1-CHECKLIST.md). Đây là bước **duy nhất** trong Luồng A cần bạn tự làm 1 việc thủ công: đăng ký tài khoản [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu), vào dataset **ERA5-Land monthly averaged data** bấm accept Terms of Use, rồi tạo file `~/.cdsapirc`:
```
url: https://cds.climate.copernicus.eu/api
key: <personal-access-token>
```
Cell dưới tự kiểm tra file này có chưa trước khi chạy gì khác.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "app").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("project root:", _root)

In [ ]:
from pathlib import Path

cdsapirc = Path.home() / '.cdsapirc'
if not cdsapirc.exists():
    raise FileNotFoundError(
        f'Chưa có {cdsapirc} — đăng ký tài khoản CDS và tạo file này trước '
        '(xem hướng dẫn ở cell markdown phía trên).'
    )
print('Tìm thấy', cdsapirc)

In [ ]:
from app.data.ingest_era5 import (
    VN_BBOX,
    build_climate_panel,
    fetch_all_years,
    load_and_convert,
    sanity_check,
)

print('Bbox tải (N, W, S, E):', VN_BBOX)

## Tải theo từng năm

Mỗi request xếp hàng ở CDS, có thể mất vài phút tới vài chục phút tuỳ tải hệ thống — **chạy cell này rồi làm việc khác trong lúc chờ**. `fetch_all_years()` tự bỏ qua năm đã tải, tự retry (3 lần, backoff) nếu 1 request lỗi, và KHÔNG dừng cả loạt nếu 1 năm lỗi hẳn — năm nào lỗi sẽ in ra và trả về trong `failed_years` để biết chạy bù sau. Đổi `YEARS` thành khoảng nhỏ (vd `range(2020, 2021)`) để test nhanh trước.

In [ ]:
YEARS = range(1994, 2026)

failed_years = fetch_all_years(YEARS)
print(f"\nXong. {len(failed_years)} năm lỗi hẳn: {failed_years}")

## Sanity check đơn vị (1 năm mẫu)

Nhiệt độ VN phải 5-40°C, độ ẩm 30-100%, mưa ≥ 0. Sai khoảng này gần như chắc chắn quên đổi Kelvin hoặc quên nhân số ngày trong tháng cho mưa.

In [ ]:
from app.data.ingest_era5 import RAW_DIR

sample_year = max(y for y in YEARS if y not in failed_years)
sample_ds = load_and_convert(RAW_DIR / f'era5_land_monthly_{sample_year}.nc')
sanity_check(sample_ds)

## Zonal stats: raster → (tỉnh, tháng)

v1 = trung bình theo diện tích đơn thuần (`stat='mean'`), không trọng số dân số — xem PHASE-1-CHECKLIST §A4 cho nâng cấp v2.

In [ ]:
import matplotlib.pyplot as plt

climate_panel = build_climate_panel(YEARS)
climate_panel.head()

## Sanity check zonal (1 tỉnh, chuỗi thời gian)

Nhiệt độ trung bình tháng của 1 tỉnh phải có dao động mùa vụ rõ (nóng hơn giữa năm), không phẳng lì (dấu hiệu zonal stat bị lỗi, toàn NaN hoặc toàn 1 giá trị).

In [ ]:
sample_province = climate_panel['province_id'].iloc[0]
series = climate_panel[climate_panel['province_id'] == sample_province].sort_values('month')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series['month'], series['temp_mean'])
ax.set_title(f'Nhiệt độ trung bình tháng — {sample_province}')
plt.show()

assert climate_panel['temp_mean'].between(5, 40).mean() > 0.95, 'Quá nhiều giá trị nhiệt độ ngoài khoảng hợp lý'
print('Sanity check pass.')

## Lưu ra `data/interim/`

In [ ]:
interim_dir = _root / 'data' / 'interim'
interim_dir.mkdir(parents=True, exist_ok=True)
out_path = interim_dir / 'climate_by_province_month.parquet'
climate_panel.to_parquet(out_path, index=False)
print(f'Đã lưu {len(climate_panel)} dòng vào {out_path}')

**Tiếp theo:** sau khi cả 3 notebook (00, 01, 02) đã lưu xong `data/interim/*.parquet`, chạy từ thư mục `ai-service/`:
```bash
python -m app.data.build_panel
```
`VERSION` tự lên `0.2.0` khi phát hiện đủ cả 3 nguồn — xem `app/data/build_panel.py`.